# AI Lab 2: Multiple Object Video Detection (UCU AI 2026)
## Checkpoint 5: Query-Based Detector (RT-DETR / Deformable DETR) (0.75 pt)

This notebook delivers **Part 5: Query-Based Object Detection** for UCU AI 2026 Computer Vision Homework 2 on the VisDrone aerial video dataset.

### Objectives Covered in this Checkpoint:
1. **Query-Based Architecture Setup**:
 - Train a query-based transformer detector (e.g. RT-DETR or Deformable DETR) under a comparable compute budget.
2. **Theoretical Analysis: Set Prediction & Bipartite Matching**:
 - Explain how learnable object queries and bipartite Hungarian matching eliminate the need for Non-Maximum Suppression (NMS).
 - Analyze how direct set prediction changes loss gradients and training convergence dynamics.
3. **Hyperparameter & Architectural Tuning**:
 - **Investigation 1**: Number of object queries scaled relative to VisDrone scene density ($300 \to 500+$ queries).
 - **Investigation 2**: Denoising task / auxiliary losses and matching cost weight balance ($L_1$, GIoU, Focal class cost).
4. **Validation Benchmark & Convergence Analysis**:
 - Benchmark validation metrics ($mAP@[0.50:0.95]$, $AP_{50}$, $AP_{small}$, FPS) and compare convergence vs. CNN detectors.
5. **Diagnostic Failure Cases Analysis**:
 - Visual and quantitative breakdown of three distinct error modes in dense aerial imagery.
---

## 1. Environment Setup & Dataset Verification
> Initializes hardware profile, mounts Google Drive for persistent checkpointing, and verifies dataset presence.

In [ ]:
import sys
import os
from pathlib import Path
import torch

# 1. Environment & Hardware Initialization
from src.setup_utils import setup_environment, ensure_dataset

env = setup_environment(seed=42, mount_gdrive=True)
device = env["device"]
hw_cfg = env["hw_cfg"]
gdrive_dir = env["gdrive_dir"]

# 2. Dataset Path Resolution & Automatic Download
dataset_dir, coco_train, coco_val, class_names = ensure_dataset()
print(f"Dataset root: {dataset_dir}")
print(f"Target classes ({len(class_names)}): {class_names}")

## 2. Production Preprocessing & PyTorch DataLoaders
> Configures sequence-isolated DataLoaders (stride $k=3$) and standardized COCOeval metric harness.

In [ ]:
# Build sequence-isolated PyTorch DataLoaders
from src.dataset import VisDroneDataset, build_dataloader
from pycocotools.coco import COCO
from src.evaluation import evaluate_coco_metrics

train_loader = build_dataloader(
    dataset_dir=dataset_dir,
    split="train",
    target_size=hw_cfg.get("target_size", (640, 640)),
    batch_size=hw_cfg.get("batch_size", 2),
    frame_stride=3,
    num_workers=hw_cfg.get("num_workers", 0),
    is_train=True,
)

val_loader = build_dataloader(
    dataset_dir=dataset_dir,
    split="val",
    target_size=hw_cfg.get("target_size", (640, 640)),
    batch_size=hw_cfg.get("val_batch_size", 4),
    frame_stride=3,
    num_workers=hw_cfg.get("num_workers", 0),
    is_train=False,
)

coco_val = COCO(str(dataset_dir / "val" / "annotations_coco.json"))
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 3. Theoretical Analysis: Query-Based Paradigm

#### A. Set Prediction & Bipartite Matching vs. NMS:
* **Conventional Detectors (Checkpoints 2, 3, 4)**:
 - Generate a massive surplus of redundant candidate boxes across feature grids.
 - Rely on hand-crafted **Non-Maximum Suppression (NMS)** with an arbitrary IoU threshold (e.g. 0.5) to discard duplicate detections.
 - *Failure mode in VisDrone*: In dense traffic or walking crowds, real adjacent pedestrians overlap and are erroneously suppressed by NMS.
* **Query-Based Detectors (DETR / RT-DETR)**:
 - Formulates detection as a **direct set prediction problem**.
 - A fixed number of learnable positional queries ($N$) attend to multi-scale image features via transformer cross-attention.
 - Uses the **Hungarian Algorithm** to compute an optimal one-to-one bipartite matching between the $N$ predicted boxes and $M$ ground-truth targets ($N \ge M$):
 $$\hat{\sigma} = \arg\min_{\sigma \in \mathfrak{S}_N} \sum_{i=1}^N \mathcal{L}_{\text{match}}(y_i, \hat{y}_{\sigma(i)})$$
 - Each target object is assigned to **exactly one query**. Duplicate predictions receive no matching reward and are naturally penalized, **eliminating NMS completely**.

#### B. Hyperparameter Investigations for Dense Aerial Imagery:
1. **Investigation 1: Number of Queries vs. Scene Density**:
 - Standard DETR uses 100 queries (sufficient for COCO, where images average ~7 objects).
 - In VisDrone, dense frames contain up to 100+ objects. Scaling queries to **300500** ensures sufficient query capacity to capture every small target.
2. **Investigation 2: Denoising Task & Multi-Scale Attention**:
 - Early DETR training suffers from unstable bipartite matching.
 - Contrastive denoising training (DN-DETR / RT-DETR) feeds noisy ground-truth boxes to stabilize gradient flow and accelerate convergence within budget.

## 4. Model Instantiation & Training (Boilerplate)
> Instantiate your query-based detector (e.g., RT-DETR via Ultralytics / Hugging Face / torchvision) and configure checkpointing.

In [ ]:
# Query-Based Model Setup Boilerplate Placeholder
# e.g., RT-DETR (Real-Time DEtection TRansformer)
checkpoint_dir = (
    gdrive_dir if gdrive_dir is not None else Path("checkpoints/query_based")
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints will be saved to: {checkpoint_dir}")
print("Ready for query-based model training execution.")

## 5. Standardized Validation Evaluation & Model Comparison
> Evaluates validation metrics and benchmarks against CNN-based detectors.

In [ ]:
# Validation Evaluation Placeholder
print("=" * 76)
print("CHECKPOINT 5: QUERY-BASED DETECTOR VALIDATION REPORT")
print("=" * 76)
print(" mAP @[0.50:0.95] : [Pending Training Execution]")
print(" AP @0.50 : [Pending Training Execution]")
print(" AP Small : [Pending Training Execution]")
print(" Inference Latency: [Pending Benchmark]")
print(" Inference Speed : [Pending Benchmark] FPS")
print("=" * 76)

### Master Detection Paradigm Comparison

| Paradigm | Model | Parameters | $mAP@[0.50:0.95]$ | $AP_{50}$ | $AP_{small}$ | Speed (FPS) | NMS Required? |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Two-Stage Anchor-Based** | Faster R-CNN ResNet-50-FPN V2 | 43.3M | **0.0475** | **0.1489** | 0.0059 | 2.9 FPS | Yes |
| **One-Stage Anchor-Based** | YOLOv8 Nano | 3.2M | *(from CP3)* | *(from CP3)* | *(from CP3)* | ~40 FPS | Yes |
| **One-Stage Anchor-Free** | *(Anchor-Free Model)* | *(TBD)* | *(from CP4)* | *(from CP4)* | *(from CP4)* | *(TBD)* | Yes |
| **Query-Based Transformer** | RT-DETR | *(TBD)* | *(from CP5)* | *(from CP5)* | *(from CP5)* | *(TBD)* | **No (NMS-Free)** |

## 6. Diagnostic Failure Case Analysis
> Analysis of three primary error modes for query-based transformer detectors on aerial video:
1. **Query Saturation in Ultra-Dense Clusters**: Scenes with >150 objects exceeding the active query budget.
2. **Small Scale Attention Attenuation**: Multi-head self-attention diluting features for sub-16px objects across global context.
3. **Slow Early-Stage Bipartite Matching Jitter**: Oscillating matching assignments between adjacent objects of the same class.

In [ ]:
# Failure Case Visualization Placeholder
print("Ready for diagnostic failure case analysis and plotting.")